In [1]:
import pandas as pd

train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 24)
Validation: (14471, 24)
Test: (14472, 24)


In [2]:
for i, col in enumerate(train.columns, 1):
    print(i, col)

1 order_id
2 customer_id
3 order_status
4 order_purchase_timestamp
5 order_approved_at
6 order_delivered_carrier_date
7 order_delivered_customer_date
8 order_estimated_delivery_date
9 customer_unique_id
10 customer_zip_code_prefix
11 customer_city
12 customer_state
13 item_count
14 total_price
15 total_freight
16 payment_count
17 total_payment
18 max_installments
19 review_count
20 avg_review_score
21 unique_products
22 unique_sellers
23 unique_categories
24 is_late


In [3]:
target = "is_late"

drop_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "review_count",
    "avg_review_score"
]

X_train = train.drop(columns=drop_columns + [target])
y_train = train[target]

X_val = validation.drop(columns=drop_columns + [target])
y_val = validation[target]

X_test = test.drop(columns=drop_columns + [target])
y_test = test[target]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

X_train: (67533, 14)
X_val: (14471, 14)
X_test: (14472, 14)

Features:
['order_purchase_timestamp', 'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'max_installments', 'unique_products', 'unique_sellers', 'unique_categories']


In [4]:
for df in [X_train, X_val, X_test]:
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

    # Purchase time features
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    # Number of days promised for delivery
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    # Remove original datetime columns
    df.drop(
        columns=[
            "order_purchase_timestamp",
            "order_estimated_delivery_date"
        ],
        inplace=True
    )

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

X_train: (67533, 17)
X_val: (14471, 17)
X_test: (14472, 17)

Features:
['customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'max_installments', 'unique_products', 'unique_sellers', 'unique_categories', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_days']


In [5]:
# Check column types
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

print("\nMissing values:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

Categorical columns:
['customer_city', 'customer_state']

Numerical columns:
['customer_zip_code_prefix', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'max_installments', 'unique_products', 'unique_sellers', 'unique_categories', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_days']

Missing values:
payment_count       1
total_payment       1
max_installments    1
dtype: int64


In [6]:
payment_cols = [
    "payment_count",
    "total_payment",
    "max_installments"
]

for df in [X_train, X_val, X_test]:
    df[payment_cols] = df[payment_cols].fillna(0)

print("Train missing:", X_train.isnull().sum().sum())
print("Validation missing:", X_val.isnull().sum().sum())
print("Test missing:", X_test.isnull().sum().sum())

Train missing: 0
Validation missing: 0
Test missing: 0


In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_cols = ["customer_city", "customer_state"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (67533, 3789)
Validation: (14471, 3789)
Test: (14472, 3789)


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (67533, 3789)
Validation: (14471, 3789)
Test: (14472, 3789)


In [9]:
import joblib
import json
from scipy import sparse

# Save the fitted preprocessor
joblib.dump(preprocessor, "preprocessor.pkl")

# Save processed datasets
sparse.save_npz("X_train_processed.npz", X_train_processed)
sparse.save_npz("X_val_processed.npz", X_val_processed)
sparse.save_npz("X_test_processed.npz", X_test_processed)

# Save targets
y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# Save final feature names
feature_names = preprocessor.get_feature_names_out().tolist()

with open("feature_list.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print("Notebook 5 artifacts saved successfully!")
print("Number of final features:", len(feature_names))

Notebook 5 artifacts saved successfully!
Number of final features: 3789
